# 큐싱 탐지 — URLBERT 4종 분류 헤드 학습·평가 (Kaggle)

설계 문서: `docs/specs/2026-11-09-urlbert-model-design.md` (+ 개정 1, 개정 2)

**실행 전 준비 (오른쪽 패널)**
1. **Settings → Accelerator: GPU T4 x2 또는 P100**, **Internet: On** (휴대폰 인증 필요)
2. **Add Input**
   - `qsing-code` : 이 프로젝트의 `model` 폴더를 zip으로 올린 Kaggle Dataset (필수)
   - `sid321axn/malicious-urls-dataset` : 검색해서 추가 (필수)
   - PhiUSIIL CSV : UCI에서 받아 Dataset으로 업로드 (권장)
   - 외부 최신 데이터 : URLhaus CSV, OpenPhish feed txt(여러 개 가능), Tranco top-1m.csv (선택)
3. 위에서부터 **Run All** 하면 됩니다. 리허설(작은 표본) → 본 실행 순서로 진행합니다.

결과물: `/kaggle/working/qsing_artifacts.zip` (서버용), `/kaggle/working/qsing_out/reports/` (지표·리포트)

In [ ]:
# 0. 코드 준비 + 패키지 확인
import os, sys, glob, shutil, subprocess, importlib

hits = glob.glob('/kaggle/input/**/qsing_model/__init__.py', recursive=True)
if not hits:  # zip이 풀리지 않은 채 올라간 경우
    import zipfile
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        with zipfile.ZipFile(z) as zf:
            if any(n.endswith('qsing_model/__init__.py') for n in zf.namelist()):
                zf.extractall('/kaggle/working/_code_unzip')
    hits = glob.glob('/kaggle/working/_code_unzip/**/qsing_model/__init__.py', recursive=True)
assert hits, "qsing_model 코드를 찾지 못했습니다. 'qsing-code' 데이터셋(model 폴더 zip)을 Add Input으로 붙였는지 확인하세요."
src_root = os.path.dirname(os.path.dirname(hits[0]))
CODE = '/kaggle/working/qsing_code'
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(src_root, CODE)
sys.path.insert(0, CODE)
print('코드 위치:', CODE)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tldextract'], check=False)
import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version('4.48.0'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.48'], check=True)
    print('transformers를 업그레이드했습니다. 메뉴 Run → Restart & Run All 로 다시 실행하세요.')
import torch, tldextract
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)

In [ ]:
# 0-1. 단위 테스트 (URL 파싱·정제·분리·지표·임계값·추론 모듈·파이프라인 리허설)
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', os.path.join(CODE, 'tests')], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, '단위 테스트 실패 — 위 메시지를 확인하세요.'

## 설정 — 입력 파일 자동 탐색

In [ ]:
import datetime
from qsing_model import pipeline as P
from qsing_model import data as D

cfg = P.Config(
    out_dir='/kaggle/working/qsing_out',
    collected_at=datetime.date.today().isoformat(),   # 외부 데이터를 다른 날 받았다면 그 날짜로 바꾸세요
    malicious_urls_csv=D.find_file(['malicious_phish*.csv', '*malicious*url*.csv']),
    phiusiil_csv=D.find_file(['phiusiil*.csv', '*phiusiil*'], required=False),
    urlhaus_csv=D.find_file(['*urlhaus*.csv', '*urlhaus*.txt', 'csv.txt'], required=False),
    openphish_txts=D.find_files(['*openphish*.txt', 'feed*.txt']),
    phishtank_csv=D.find_file(['*phishtank*.csv', 'online-valid*.csv'], required=False),
    tranco_csv=D.find_file(['*tranco*.csv', 'top-1m.csv'], required=False),
    # baseline_label_map={0: 'benign', 1: 'defacement', 2: 'phishing', 3: 'malware'},  # 자동 매핑 실패 시에만
)
for k in ('malicious_urls_csv', 'phiusiil_csv', 'urlhaus_csv', 'openphish_txts', 'phishtank_csv', 'tranco_csv'):
    print(f'{k:20s}', getattr(cfg, k))

## P0 환경 확인 + 원격 코드 검토
제공 헤드는 `trust_remote_code=True`로만 로드됩니다(저장소의 파이썬 코드 실행).
아래에 출력되는 코드를 읽어보고 이상한 동작(파일 쓰기, 네트워크 전송 등)이 없는지 확인한 뒤 진행하세요.
저장소 커밋은 여기서 해시로 고정됩니다.

In [ ]:
rep0 = P.p0_environment(cfg, show_code=True)

## 리허설 — 작은 표본으로 전 과정 점검 (약 수 분)
본 실행 전에 P1~P7을 6천 건 정도로 한 번 돌려 오류를 미리 잡습니다. 결과는 `qsing_out/_smoke`에 따로 저장되며 본 결과에 섞이지 않습니다.

In [ ]:
s = P.smoke_config(cfg)
s.revision = cfg.revision
P.p1_prepare(s); P.p2_split(s); P.p3_baseline(s); P.p4_embed(s); P.p4_train(s); P.p6_thresholds(s); P.p5_evaluate(s)
P.p7_export(s, zip_path='/kaggle/working/_smoke_artifacts')
print('\n리허설 통과 — 본 실행으로 진행합니다.')

## P1 데이터 점검·전처리

In [ ]:
rep1 = P.p1_prepare(cfg)

## P2 도메인 그룹 분리 (70:15:15) + 누수 검사

In [ ]:
rep2 = P.p2_split(cfg)

## P3 기준선 — 제공 `malicious_urls_4class` 헤드 (학습 데이터와 겹쳐 점수가 부풀려졌을 수 있음)

In [ ]:
base = P.p3_baseline(cfg)

## P4 임베딩 계산 + 헤드 학습 ({raw, strip} × {Linear, MLP}, 선택 기준 = val Macro F1 최저값)

In [ ]:
rep4 = P.p4_embed(cfg)

In [ ]:
sel = P.p4_train(cfg)

## P6 위험도 임계값 (val 세트로만 결정) — P5의 '임계값 기준 오탐·미탐률'에 쓰이므로 P5보다 먼저 실행

In [ ]:
thr = P.p6_thresholds(cfg)

## P5 평가 — test 세트, PhiUSIIL, 외부 최신 데이터, 기준선 비교

In [ ]:
metrics = P.p5_evaluate(cfg)
from IPython.display import Image, display
display(Image(cfg.path('reports', 'confusion_matrix.png')))

## P7 서버용 산출물 내보내기 + CPU 추론 시간 측정

In [ ]:
exp = P.p7_export(cfg)

## 결과 요약 — 아래 출력을 `work_history.md` 맨 아래에 붙여넣으세요

In [ ]:
print(P.run_summary_md(cfg))
shutil.make_archive('/kaggle/working/qsing_reports', 'zip', cfg.path('reports', '.'))
print('다운로드: /kaggle/working/qsing_artifacts.zip (서버용), /kaggle/working/qsing_reports.zip (지표·리포트)')